# How to choose and configure the Gibbs sampler

`fit()` already uses Gibbs for the Gaussian spatial models — SAR, SEM, SDM,
SDEM and the Gaussian panel families — because they have a registered Gibbs
sampler and `sampler=None` prefers it. This guide covers the decisions left to
you: confirming what you got, checking it against NUTS, picking a backend,
loosening the ρ update when it mixes badly, and changing how the
log-determinant is computed.

For the block structure and the full table of backends and log-determinant
methods, see [Supported Models](../models.md); for why the conjugate blocks
help at all, [Architecture](../architecture.md).

In [ ]:
import time

import arviz as az
import geopandas as gpd
import libpysal
import numpy as np
import pandas as pd

from neighbayes.models import SAR, SDEM, SDM, SEM

gdf = gpd.read_file(libpysal.examples.get_path("columbus.shp"))
y = gdf["CRIME"].values.astype(float)
X = np.column_stack(
    [
        np.ones(len(y)),
        gdf["INC"].values.astype(float),
        gdf["HOVAL"].values.astype(float),
    ]
)
W = libpysal.graph.Graph.build_contiguity(gdf).transform("r")

COMMON = dict(draws=2000, tune=1000, chains=4, random_seed=42)


def fit_timed(model, **kw):
    start = time.perf_counter()
    idata = model.fit(progressbar=False, **kw)
    return idata, time.perf_counter() - start


def efficiency(label, idata, secs, params=("rho", "sigma")):
    ess = az.ess(idata, var_names=list(params))
    rhat = az.rhat(idata, var_names=list(params))
    row = {"seconds": secs}
    for p in params:
        row[f"ess {p}"] = float(ess[p])
        row[f"ess/sec {p}"] = float(ess[p]) / secs
    row["max rhat"] = float(max(float(rhat[p]) for p in params))
    return pd.Series(row, name=label)


print(f"n = {len(y)}, k = {X.shape[1]}")

## Confirm which sampler ran

Nothing in `fit()` announces its choice, so check the `sample_stats` group.
A NUTS run carries `tree_depth` and `diverging`; a Gibbs run does not.

In [ ]:
idata_default, secs_default = fit_timed(SAR(y=y, X=X, W=W), **COMMON)

nuts_only = {"tree_depth", "diverging", "step_size"}
present = nuts_only & set(idata_default.sample_stats.data_vars)
print("sample_stats:", sorted(idata_default.sample_stats.data_vars))
print("ran NUTS" if present else "ran Gibbs")

## Check the Gibbs result against NUTS

Worth doing once on any new dataset. The two samplers target the same
posterior, so a disagreement beyond Monte Carlo error is a defect rather than a
tuning difference — and this is the cheapest validation available.

In [ ]:
idata_nuts, secs_nuts = fit_timed(
    SAR(y=y, X=X, W=W), sampler="nuts", target_accept=0.9, **COMMON
)

pd.DataFrame(
    {
        "Gibbs": [
            float(idata_default.posterior["rho"].mean()),
            float(idata_default.posterior["sigma"].mean()),
        ],
        "NUTS": [
            float(idata_nuts.posterior["rho"].mean()),
            float(idata_nuts.posterior["sigma"].mean()),
        ],
    },
    index=["rho", "sigma"],
).round(4)

In [ ]:
pd.DataFrame(
    [
        efficiency("Gibbs", idata_default, secs_default),
        efficiency("NUTS", idata_nuts, secs_nuts),
    ]
).round(3)

Read the **ESS-per-second** column, not `seconds` — a sampler twice as
fast and half as efficient has bought you nothing. Columbus has $n = 49$; for
timings at realistic sizes see
[Performance & Profiling](../performance/index.md).

Two arguments do not cross the sampler boundary: `target_accept` raises
`TypeError` under Gibbs, and `idata_kwargs={"log_likelihood": True}` is
NUTS-only — Gibbs builds that group unasked.

In [ ]:
try:
    SAR(y=y, X=X, W=W).fit(sampler="gibbs", target_accept=0.9, draws=10, tune=10)
except TypeError as err:
    print(f"TypeError: {err}")

## Pin a backend

`gibbs_backend` defaults to `"auto"`, which takes JAX when it is installed and
falls back to NumPy. Pin it explicitly when you are benchmarking, or when you
want the NumPy path's process-level parallelism (`n_jobs`) instead of the JAX
path's vectorised chains (`chain_method`).

In [ ]:
pd.DataFrame(
    [
        efficiency(
            backend,
            *fit_timed(
                SAR(y=y, X=X, W=W), sampler="gibbs", gibbs_backend=backend, **COMMON
            ),
        )
        for backend in ("numpy", "jax")
    ]
).round(3)

JAX pays a one-off compilation cost on the first sweep, so a single
small-$n$ timing is the wrong basis for choosing — at $n = 49$ that cost is a
visible fraction of the total, and at the sizes where you would reach for JAX
it is amortized to nothing.

## Loosen the ρ update when it mixes poorly

If `ess_bulk` on ρ or λ is low while β and σ² look fine, the slice sampler's
interval is the thing to change. `slice_width` sets its starting width; the
sampler adapts from there during warmup.

In [ ]:
idata_tuned, secs_tuned = fit_timed(
    SAR(y=y, X=X, W=W),
    sampler="gibbs",
    gibbs_backend="numpy",
    slice_width=0.25,
    n_jobs=-1,
    thin=2,
    **COMMON,
)
print(f"draws kept per chain with thin=2: {idata_tuned.posterior.sizes['draw']}")
efficiency("slice_width=0.25, thin=2", idata_tuned, secs_tuned).round(3)

## Change the log-determinant method

`logdet_method` goes on the **model**, not on `fit()`. Left at `None` it is
chosen for you by size, by whether $W$ is symmetric, and by a fill-in estimate;
override it when you know something the selector does not — that your graph
factorizes cheaply, or that it does not.

In [ ]:
# The constructor reports the valid names, so this cannot drift out of date.
try:
    SAR(y=y, X=X, W=W, logdet_method="list-them-please")
except ValueError as err:
    print(err)

**Verify any approximate method against an exact one before trusting it.**
On a small problem the exact answer is free, so there is no excuse not to.

In [ ]:
pd.DataFrame(
    [
        pd.Series(
            {
                "rho mean": float(idata.posterior["rho"].mean()),
                "rho sd": float(idata.posterior["rho"].std()),
                "seconds": secs,
                "ess rho": float(az.ess(idata, var_names=["rho"])["rho"]),
            },
            name=method,
        )
        for method, (idata, secs) in (
            (
                m,
                fit_timed(
                    SAR(y=y, X=X, W=W, logdet_method=m), sampler="gibbs", **COMMON
                ),
            )
            for m in ("eigenvalue", "chebyshev", "cheb_stochastic")
        )
    ]
).round(4)

`eigenvalue` is exact at this size, so it is the reference. A method whose
ρ posterior drifts from it is trading accuracy for speed — sometimes the right
trade at scale, never one to make unknowingly.

## Apply it to SEM, SDM and SDEM

The same call works for all four Gaussian models; only the name of the spatial
parameter changes (`rho` for the lag models, `lam` for the error models).

In [ ]:
pd.DataFrame(
    [
        pd.Series(
            {
                "spatial parameter": param,
                "posterior mean": float(idata.posterior[param].mean()),
                "n coefficients": idata.posterior["beta"].sizes["coefficient"],
                "ess": float(az.ess(idata, var_names=[param])[param]),
                "rhat": float(az.rhat(idata, var_names=[param])[param]),
            },
            name=name,
        )
        for name, param, idata in (
            (n, p, fit_timed(cls(y=y, X=X, W=W), sampler="gibbs", **COMMON)[0])
            for n, cls, p in (
                ("SAR", SAR, "rho"),
                ("SDM", SDM, "rho"),
                ("SEM", SEM, "lam"),
                ("SDEM", SDEM, "lam"),
            )
        )
    ]
).round(4)

## What to check before trusting the output

`ess_bulk` and `r_hat` on the spatial parameter first — it is the only
non-conjugate block, so it is where trouble shows up.

In [ ]:
az.summary(idata_default, var_names=["rho", "sigma"]).round(4)

:::{caution} Do not select a specification with LOO or WAIC
Both run on this output and both mislead for spatial autoregressive models.
Pointwise leave-one-out assumes observations are exchangeable given the
parameters; under a spatial lag they are not, because dropping $y_i$ changes
the implied mean of its neighbours through $(I - \rho W)^{-1}$.

Use [Bayesian LM specification tests](bayesian_lmtests.ipynb) or
[spatial block cross-validation](spatial_cv_demo.ipynb) instead.
:::

In [ ]:
model_diag = SAR(y=y, X=X, W=W)
model_diag.fit(sampler="gibbs", progressbar=False, **COMMON)
model_diag.spatial_diagnostics_decision()

## See also

- [Supported Models](../models.md) — the block structure, every backend, and
  all eleven log-determinant methods
- [Architecture](../architecture.md) — why conjugacy beats gradients here
- [How to set priors](priors.ipynb) — the σ² prior is what keeps that block
  conjugate
- [How to run Bayesian LM specification tests](bayesian_lmtests.ipynb) —
  choosing *which* model to fit, before tuning how you fit it
- [Performance & Profiling](../performance/index.md) — Gibbs-versus-NUTS and
  backend timings at realistic $n$